# Sales ETL Pipeline — Data Quality & Consolidation

**Contexto:** Pipeline de ingestão, limpeza e validação de registros brutos oriundos do sistema transacional de vendas. O objetivo é consolidar um dataset confiável para consumo analítico, eliminando anomalias estruturais e violações de regras de negócio antes da camada de relatórios financeiros.

**Stack:** Python 3 — sem dependências externas (stdlib only).

---

## 1. Ingestão de Dados Brutos (Raw Data)

Simulação da extração de registros do sistema de vendas. O dataset bruto contém inconsistências estruturais deliberadas — preços negativos, quantidades fora do range aceitável, strings com whitespace e campos nulos — que serão tratadas nas etapas subsequentes do pipeline.

In [1]:
# Conjunto de registros brutos extraídos do sistema transacional de vendas.
# Este dataset contém anomalias intencionais para validação do pipeline de limpeza.
vendas = [
    {"produto": "mouse gamer", "preco": 55.00, "quantidade": 3, "vendedor": "  MARIA SILVA  "},
    {"produto": "TECLADO MECANICO", "preco": -150.00, "quantidade": 2, "vendedor": "joão santos"},
    {"produto": "  Monitor LG  ", "preco": 890.00, "quantidade": 0, "vendedor": "Ana Costa"},
    {"produto": "Mouse Gamer", "preco": 55.00, "quantidade": 5, "vendedor": "MARIA SILVA"},
    {"produto": "webcam", "preco": 0, "quantidade": 2, "vendedor": "  bruno lima  "},
    {"produto": "MOUSE GAMER", "preco": 55.00, "quantidade": 1, "vendedor": "Maria Silva  "},
    {"produto": "monitor lg", "preco": 890.00, "quantidade": 1000, "vendedor": "Pedro Souza"},
    {"produto": "teclado mecanico", "preco": 150.00, "quantidade": 3, "vendedor": "João Santos"},
    {"produto": "", "preco": 100.00, "quantidade": 1, "vendedor": "Carlos Lima"},
    {"produto": "SSD 480GB", "preco": 280.00, "quantidade": -2, "vendedor": ""},
    {"produto": "  WEBCAM  ", "preco": 450.00, "quantidade": 2, "vendedor": "Ana Costa"},
    {"produto": "ssd 480gb", "preco": 280.00, "quantidade": 4, "vendedor": "  CARLOS LIMA  "}
]

print(f"[LOG] Total de registros extraídos: {len(vendas)}\n")
print("[LOG] Amostra dos dados brutos (Raw):")
for i, venda in enumerate(vendas[:3], 1):
    print(f"  {i}. Produto: '{venda['produto']}' | Preco: R$ {venda['preco']} | Qtd: {venda['quantidade']} | Vendedor: '{venda['vendedor']}'")

[LOG] Total de registros extraídos: 12

[LOG] Amostra dos dados brutos (Raw):
  1. Produto: 'mouse gamer' | Preco: R$ 55.0 | Qtd: 3 | Vendedor: '  MARIA SILVA  '
  2. Produto: 'TECLADO MECANICO' | Preco: R$ -150.0 | Qtd: 2 | Vendedor: 'joão santos'
  3. Produto: '  Monitor LG  ' | Preco: R$ 890.0 | Qtd: 0 | Vendedor: 'Ana Costa'


## 2. Data Cleaning — Transformação e Padronização

Definição dos módulos de limpeza responsáveis pela padronização de strings. Cada função aplica trim de whitespace e normalização de case para garantir integridade referencial no dataset consolidado.

### 2.1 Normalização de Nomes de Vendedores

Remoção de espaços laterais e conversão para Title Case, assegurando consistência para joins e agrupamentos por vendedor.

In [2]:
def limpar_nome(nome):
    """Remove whitespace lateral e aplica Title Case para padronização de nomes de vendedores."""
    nome_sem_espacos = nome.strip()
    nome_formatado = nome_sem_espacos.title()
    return nome_formatado

print("[LOG] Testando módulo de limpeza de nomes...")
testes_nome = [("  MARIA SILVA  ", "Maria Silva"), ("joão santos", "João Santos"), ("  Ana  ", "Ana")]
for entrada, esperado in testes_nome:
    resultado = limpar_nome(entrada)
    status = "PASS" if resultado == esperado else "FAIL"
    print(f"  [{status}] Entrada: '{entrada}' -> Saída: '{resultado}'")

[LOG] Testando módulo de limpeza de nomes...
  [PASS] Entrada: '  MARIA SILVA  ' -> Saída: 'Maria Silva'
  [PASS] Entrada: 'joão santos' -> Saída: 'João Santos'
  [PASS] Entrada: '  Ana  ' -> Saída: 'Ana'


### 2.2 Normalização de Nomenclatura de Produtos

Conversão para lowercase e trim de strings para eliminar duplicatas semânticas e permitir agregações corretas por SKU no relatório final.

In [3]:
def normalizar_produto(produto):
    """Remove whitespace lateral e converte para lowercase para correta agregação por SKU."""
    produto_baguncados = produto.strip()
    produtos_formatado = produto_baguncados.lower()
    return produtos_formatado

print("[LOG] Testando módulo de normalização de produtos...")
testes_produto = [
    ("  MOUSE GAMER  ", "mouse gamer"),
    ("Monitor LG", "monitor lg"),
    ("  WEBCAM  ", "webcam"),
]

for entrada, esperado in testes_produto:
    resultado = normalizar_produto(entrada)
    status = "PASS" if resultado == esperado else "FAIL"
    print(f"  [{status}] Entrada: '{entrada}' -> Saída: '{resultado}'")

[LOG] Testando módulo de normalização de produtos...
  [PASS] Entrada: '  MOUSE GAMER  ' -> Saída: 'mouse gamer'
  [PASS] Entrada: 'Monitor LG' -> Saída: 'monitor lg'
  [PASS] Entrada: '  WEBCAM  ' -> Saída: 'webcam'


## 3. Quality Gates / Validação — Aplicação das Regras de Negócio

Implementação dos filtros de validação (Quality Gates) para identificar anomalias estruturais e financeiras no dataset bruto.

**Regras de negócio aplicadas:**
- Preco estritamente maior que zero (sem registros negativos ou zerados).
- Quantidade comercializada entre 1 e 100 unidades (proteção contra outliers de lançamento).
- Campos obrigatórios `produto` e `vendedor` não podem ser nulos ou vazios.

### 3.1 Validação de Registro Individual (Sample Check)

Verificação isolada do primeiro registro para validação do gate de negócio antes da execução em lote.

In [4]:
# Seleção do primeiro registro para verificação individual dos critérios de validação.
venda_teste = vendas[0]

print("[LOG] Validando registro de amostra:")
print(f"  Produto: '{venda_teste['produto']}'")
print(f"  Preco: R$ {venda_teste['preco']}")
print(f"  Qtd: {venda_teste['quantidade']}")
print(f"  Vendedor: '{venda_teste['vendedor']}'")
print()

if venda_teste['preco'] <= 0:
    print("[FAIL] Registro rejeitado — preco zero ou negativo.")
elif venda_teste['quantidade'] < 1 or venda_teste['quantidade'] > 100:
    print("[FAIL] Registro rejeitado — quantidade fora do range aceitável (1-100).")
elif venda_teste['produto'].strip() == "":
    print("[FAIL] Registro rejeitado — campo 'produto' vazio.")
elif venda_teste["vendedor"].strip() == "":
    print("[FAIL] Registro rejeitado — campo 'vendedor' vazio.")
else:
    print("[PASS] Registro aprovado nos Quality Gates.")

[LOG] Validando registro de amostra:
  Produto: 'mouse gamer'
  Preco: R$ 55.0
  Qtd: 3
  Vendedor: '  MARIA SILVA  '

[PASS] Registro aprovado nos Quality Gates.


### 3.2 Pipeline ETL — Processamento em Lote (Batch Processing)

Execução do pipeline completo sobre todos os registros brutos: validação, limpeza e roteamento para as partições `vendas_validas` e `vendas_invalidas`.

In [5]:
# Inicialização das partições de saída do pipeline.
vendas_validas = []
vendas_invalidas = []

for venda in vendas:
    # Aplicação sequencial dos Quality Gates de negócio.
    if venda["preco"] <= 0:
        vendas_invalidas.append(venda)
    elif venda["quantidade"] < 1 or venda["quantidade"] > 100:
        vendas_invalidas.append(venda)
    elif venda["produto"].strip() == "":
        vendas_invalidas.append(venda)
    elif venda["vendedor"].strip() == "":
        vendas_invalidas.append(venda)
    else:
        # Registro aprovado: aplica normalização antes de persistir na partição válida.
        venda_limpa = {
            "produto": normalizar_produto(venda["produto"]),
            "preco": venda["preco"],
            "quantidade": venda["quantidade"],
            "vendedor": limpar_nome(venda["vendedor"])
        }
        vendas_validas.append(venda_limpa)


print("[LOG] Aplicando Quality Gates no dataset completo...")
print()
print(f"[RESULTADO] Registros processados: {len(vendas)}")
print(f"[RESULTADO] Registros aprovados:   {len(vendas_validas)}")
print(f"[RESULTADO] Registros rejeitados:  {len(vendas_invalidas)}")
print()

if len(vendas_validas) > 0:
    print("[LOG] Amostra de registros aprovados e normalizados:")
    for i, venda in enumerate(vendas_validas[:3], 1):
        print(f"\n{i}. Produto: {venda['produto']}")
        print(f"   Preco: R$ {venda['preco']:.2f}")
        print(f"   Qtd: {venda['quantidade']}")
        print(f"   Vendedor: {venda['vendedor']}")
else:
    print("[FAIL] Nenhum registro aprovado. Verifique os Quality Gates e o dataset de entrada.")

print()

if len(vendas_invalidas) > 0:
    print("[LOG] Amostra de registros rejeitados (não processados):")
    for i, venda in enumerate(vendas_invalidas[:3], 1):
        print(f"\n{i}. {venda}")

[LOG] Aplicando Quality Gates no dataset completo...

[RESULTADO] Registros processados: 12
[RESULTADO] Registros aprovados:   6
[RESULTADO] Registros rejeitados:  6

[LOG] Amostra de registros aprovados e normalizados:

1. Produto: mouse gamer
   Preco: R$ 55.00
   Qtd: 3
   Vendedor: Maria Silva

2. Produto: mouse gamer
   Preco: R$ 55.00
   Qtd: 5
   Vendedor: Maria Silva

3. Produto: mouse gamer
   Preco: R$ 55.00
   Qtd: 1
   Vendedor: Maria Silva

[LOG] Amostra de registros rejeitados (não processados):

1. {'produto': 'TECLADO MECANICO', 'preco': -150.0, 'quantidade': 2, 'vendedor': 'joão santos'}
2. {'produto': '  Monitor LG  ', 'preco': 890.0, 'quantidade': 0, 'vendedor': 'Ana Costa'}
3. {'produto': 'webcam', 'preco': 0, 'quantidade': 2, 'vendedor': '  bruno lima  '}


## 4. Pipeline ETL — Análise Agregada

Geração de métricas analíticas a partir do dataset validado e normalizado.

### 4.1 Faturamento Total

Cálculo do faturamento bruto consolidado: somatório de `(preco x quantidade)` para cada registro aprovado nos Quality Gates.

In [6]:
print("="*80)
faturamento_total = 0
for venda in vendas_validas:
    faturamento_total = faturamento_total + (venda["preco"] * venda["quantidade"])

print(f"[RESULTADO] Faturamento Total: R$ {faturamento_total:,.2f}")

[RESULTADO] Faturamento Total: R$ 2,965.00


### 4.2 Ranking de Produtos por Volume de Transações

Contagem de ocorrências por SKU normalizado para identificar o produto com maior frequência de transações no período. Serve como insumo para análises de giro de estoque e priorização comercial.

In [7]:
print("="*80)
contagem_produtos = {}
for venda in vendas_validas:
    produto = venda["produto"]
    if produto in contagem_produtos:
        contagem_produtos[produto] = contagem_produtos[produto] + 1
    else:
        contagem_produtos[produto] = 1


# Identificação do SKU com maior volume de transações no período.
if contagem_produtos:
    produto_mais_vendido = max(contagem_produtos, key=contagem_produtos.get)
    vezes = contagem_produtos[produto_mais_vendido]
    print(f"[RESULTADO] Produto com maior volume de transacoes: {produto_mais_vendido} ({vezes} ocorrencias)")
    print()
    print("[LOG] Ranking completo de produtos por volume de transacoes:")
    for produto, qtd in sorted(contagem_produtos.items(), key=lambda x: x[1], reverse=True):
        print(f"  {produto}: {qtd} transacao(oes)")

[RESULTADO] Produto com maior volume de transacoes: mouse gamer (3 ocorrencias)

[LOG] Ranking completo de produtos por volume de transacoes:
  mouse gamer: 3 transacao(oes)
  teclado mecanico: 1 transacao(oes)
  webcam: 1 transacao(oes)
  ssd 480gb: 1 transacao(oes)


## 5. Data Quality Report

Sumário executivo do pipeline: métricas de qualidade do dataset, taxa de aprovação e indicadores financeiros consolidados. Este relatório é o output final do processo ETL.

In [8]:
print("="*80)
print("DATA QUALITY REPORT — SALES ETL PIPELINE")
print("  Tech Shop | Consolidação de Vendas do Período")
print("="*80)
print()

total_original = len(vendas)
total_validas = len(vendas_validas)
total_invalidas = len(vendas_invalidas)
taxa_qualidade = (total_validas / total_original * 100) if total_original > 0 else 0

print(f"[LOG] Registros Recebidos (Raw):    {total_original}")
print(f"[PASS] Registros Aprovados:         {total_validas} ({taxa_qualidade:.1f}%)")
print(f"[FAIL] Registros Rejeitados:        {total_invalidas} ({100-taxa_qualidade:.1f}%)")
print()

if faturamento_total > 0:
    print("="*80)
    print("ANALISE FINANCEIRA CONSOLIDADA")
    print("="*80)
    print(f"Faturamento Total:     R$ {faturamento_total:,.2f}")
    if total_validas > 0:
        ticket_medio = faturamento_total / total_validas
        print(f"Ticket Medio:          R$ {ticket_medio:,.2f}")
    print()

print("="*80)
print("[PASS] PIPELINE CONCLUIDO COM SUCESSO.")
print("="*80)

DATA QUALITY REPORT — SALES ETL PIPELINE
  Tech Shop | Consolidação de Vendas do Período

[LOG] Registros Recebidos (Raw):    12
[PASS] Registros Aprovados:         6 (50.0%)
[FAIL] Registros Rejeitados:        6 (50.0%)

ANALISE FINANCEIRA CONSOLIDADA
Faturamento Total:     R$ 2,965.00
Ticket Medio:          R$ 494.17

[PASS] PIPELINE CONCLUIDO COM SUCESSO.
